# Main planning comparison

Goal-conditioned planning on TwoRoom, Reacher, Push-T, and OGBench-Cube.
We compare SMWM with SIGReg, PLDM, DINO-WM, Random, and Forward-only under
the same evaluation protocol.

## Evaluation results

Use the [released results](../../README.md#pretrained-models), or set `RESULTS_ROOT`
to your own evaluations. Success is computed per run before aggregation across
training seeds. Random uses policy repeats.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "planning/train.py").is_file())
PLANNING = ROOT / "planning"
sys.path.insert(0, str(PLANNING))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from analysis.results import load_results, summarize
from plot_style import FULL_WIDTH, METHOD_COLORS, apply_matplotlib_style

apply_matplotlib_style()
RESULTS_ROOT = PLANNING / "outputs/eval"
OUTPUT_DIR = PLANNING / "outputs/analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

runs = load_results(RESULTS_ROOT)
summary = summarize(runs)
print(f"Loaded {len(runs)} evaluations.")

## Planning success

Mean ± sample STD over five training seeds; Random uses five policy repeats.
Each run has 100 tasks, goal offset 25, and a budget of 50 environment steps.
SMWM uses `λ=10` and predictor history 1.

Bar annotations are rounded to integers; the table shows one decimal and the CSV
retains full precision. A single available run is shown without an error bar.

In [ ]:
METHODS = {
    "smwm": "SMWM", "sigreg": "SIGReg", "pldm": "PLDM",
    "dino_wm": "DINO-WM", "random": "Random", "forward_only": "Forward-only",
}
ENVIRONMENTS = {
    "TwoRoom": "tworoom", "Reacher": "reacher",
    "Push-T": "pusht", "OGBench-Cube": "ogbcube",
}

main = summary.query("goal_offset == 25 and eval_budget == 50").copy()
main = main[main.history.isna() | main.history.eq(1)]
main = main[main.inverse_weight.isna() | main.inverse_weight.isin([0, 10])]
main = main[~main.method.eq("smwm") |
            (main.inverse_weight.eq(10) & main.sigreg_weight.eq(0))]
if main.duplicated(["environment", "method"]).any():
    raise ValueError("Several configurations remain for one method. Select one configuration per method.")

if main.empty:
    print("Download the released evaluations or run eval.py before plotting.")
else:
    main.to_csv(OUTPUT_DIR / "planning_summary.csv", index=False)
    formatted = main.assign(result=[
        f"{row.mean:.1f} ± {row.std:.1f}" if row.n > 1 else f"{row.mean:.1f}"
        for row in main.itertuples()
    ])
    table = formatted.pivot(index="environment", columns="method", values="result")
    table = table.reindex(index=ENVIRONMENTS, columns=METHODS).rename(columns=METHODS)
    table.index.name = "Environment"
    table.columns.name = None
    display(table.fillna("—").style.set_caption("Planning success (%), mean ± STD"))
    latex_table = table.fillna("—").map(lambda value: value.replace(" ± ", r" $\pm$ "))
    latex_table.to_latex(OUTPUT_DIR / "planning_summary.tex", escape=False)

    fig = plt.figure(figsize=(FULL_WIDTH, 3.85))
    grid = fig.add_gridspec(
        2, 4, height_ratios=[1, 1.15], left=0.075, right=0.995,
        bottom=0.20, top=0.94, wspace=0.18, hspace=0.08,
    )
    for column, (environment, slug) in enumerate(ENVIRONMENTS.items()):
        image_ax = fig.add_subplot(grid[0, column])
        image_ax.imshow(plt.imread(ROOT / "assets/environments" / f"{slug}.png"))
        image_ax.set_title(environment, pad=5)
        image_ax.set_axis_off()

        ax = fig.add_subplot(grid[1, column])
        rows = main[main.environment.eq(environment)].set_index("method").reindex(METHODS)
        x = np.arange(len(METHODS))
        fills, edges = zip(*(METHOD_COLORS[method] for method in METHODS))
        bars = ax.bar(x, rows["mean"], width=0.7, color=fills, edgecolor=edges, linewidth=0.8)
        has_error = rows.n.gt(1) & rows["std"].notna()
        ax.errorbar(x[has_error], rows.loc[has_error, "mean"],
                    yerr=rows.loc[has_error, "std"], fmt="none",
                    ecolor="0.35", capsize=2, capthick=0.7, linewidth=0.7)
        for bar, value, edge in zip(bars, rows["mean"], edges):
            if np.isfinite(value):
                y = value / 2 if value >= 18 else value + 5
                ax.text(bar.get_x() + bar.get_width() / 2, y, f"{value:.0f}",
                        ha="center", va="center" if value >= 18 else "bottom",
                        fontsize=7, color=edge, fontweight="bold")
        ax.set(ylim=(0, 105), yticks=[0, 50, 100], xticks=x)
        ax.set_xticklabels(METHODS.values(), rotation=55, ha="right",
                           rotation_mode="anchor", fontsize=7.5)
        ax.grid(axis="y")
        ax.margins(x=0.04)
        if column == 0:
            ax.set_ylabel("Success rate (%)")
        else:
            ax.tick_params(axis="y", labelleft=False, length=0)
    fig.savefig(OUTPUT_DIR / "planning_comparison.pdf")
    plt.show()